# D1-S2-Nb3: The Powers of Low-Level Probing
Showcase:
- state-verification
- auto-correction / self-healing aPs

## Set your `protocol_mode`

...so it is easy to switch between simulation and execution

In [1]:
prep_simulation = True  # True or False

star_simulation = True  # True or False

run_identifier = "nb3"

## Library Import

In [2]:
import asyncio
import logging
import time

---
## Visualizer3D
...so everything that follows has one highest-level reference frame

In [ ]:
from pylabrobot.resources import Coordinate, Resource
from pylabrobot.visualizer3D.facility import Facility
from pylabrobot.visualizer3D.server import Viewer3D

facility = Facility(name="facility", size_x=2_400, size_y=1_000, size_z=0)

viewer = Viewer3D(root = facility, name="nb3_probing")

await viewer.start()

viewer on http://127.0.0.1:1339/#token=pd_-DaD3Khp5ptWevAgchoq6e6RqpB_oz-toXEhejGk  (websocket 2123)


viewer: a browser connected, drawing with WebGPU


2026-09-27 13:23:19,481 - pylabrobot.visualizer3D.server - WARNING - refused a websocket without this viewer's token: a page left open from an earlier viewer, or one opened without the link it printed. Repeats are logged at debug.


## Device setups
### (shadow) STAR(let)

In [ ]:
from pylabrobot.hamilton.star import STARlet

star = STARlet(simulation=star_simulation)

facility.assign_child_resource(
    resource=star,
    location=Coordinate(x=800.0, y=0.0, z=0.0)
)

await star.setup()

2026-09-27 13:23:18,225 - pylabrobot.hamilton.star.driver.master - INFO - [Hamilton STAR] Connected on simulation (no link)
  Firmware: master 7.6S 25 2021_11_05 (GRU C0), pipettes 4.0S j 2022-03-16, x_arm 1.4S 2012-04-25, head96 5.0S i 2021-10-22 (H0 XE167), iswap 4.1S 2011-12-19, autoload 3.4S f 2017-01-09
  Arms: 1
  Configuration: 30 slots
  Autoload: 1D barcode scanner
    left: hamilton_legacy_star_dual_rail_arm, 354.0 mm wide, travel 95.0 to 800.2 mm, workspace -323.2 to 977.2 mm
      channels: 8 (1000uL) | 96-head: 96 head II | 384-head: none | iSWAP: wide gripper


### Prep

In [5]:
from pylabrobot.hamilton.prep import Prep

prep = Prep(
    simulation=prep_simulation,
    )

facility.assign_child_resource(
    prep,
    location=Coordinate(x=0.0, y=0.0, z=0.0)
)

await prep.setup()

2026-09-27 13:23:18,532 - pylabrobot.hamilton.prep.driver.master - INFO - tips held at setup: none
2026-09-27 13:23:18,533 - pylabrobot.hamilton.prep.driver.master - INFO - [Hamilton Prep] Connected on simulation (no link)

  Serial: PRPAA1087

  Firmware: MLPrep Runtime V1.2.2.444 99020-02 Rev G

  Configuration: 
 - enclosure none,  - safe speeds off,
 - traverse height 167.5 mm
  Deck: x 0.00 to 299.00 mm, y -9.00 to 385.00 mm, z 18.03 to 167.50 mm; 3 sites, 3 waste sites
  Pipettes: 2, v2 aspirate/dispense
    channel 0 (rear): firmware Channel1ml Runtime V1.4.9.249 98228-02 Rev K, x 0.19 to 299.19 mm, y 0.00 to 385.00 mm, z 18.03 to 167.50 mm
    channel 1 (front): firmware Channel1ml Runtime V1.4.9.249 98228-02 Rev K, x 0.19 to 299.19 mm, y -9.00 to 376.00 mm, z 18.03 to 167.50 mm
  8-channel head: none


---
## Pickup probing tools
...all Hamilton liquid handling workstations come with "teaching needles" / metal tips attached to some part of the deck (usually the waste block)

you can easily find them by using PLR's `get_resource()` finding function:

In [6]:
prep_tipspot = prep.get_resource("Prep_teaching_needle")
prep_tipspot

TipSpot(name='Prep_teaching_needle', location=Coordinate(002.510, 218.540, 075.750), size_x=6.0, size_y=6.0, size_z=0.0, category=teaching_needle)

On the STAR there is not just *one* teaching needle, but an entire tiprack of teaching needles:

In [7]:
star_teaching_tiprack = star.get_resource("STARlet_deck_teaching_tip_rack")
star_teaching_tiprack

TipRack(name='STARlet_deck_teaching_tip_rack', size_x=9, size_y=72, size_z=50.4, location=Coordinate(005.900, 346.100, 000.000))

Having the tipspots/tiprack readily available we can pick them up:

In [8]:
prep_channel_idx = 1 # easier to see for you guys in the audience ;)

await prep.pipettes.pick_up_tips(
    tip_spots=[prep_tipspot], # has to be a list
    use_channels=[prep_channel_idx]
)

Since, even on the Prep, there are multiple pipettes - and we *could* therefore pick up multiple needles/tips - arguments always take a list.

The index of an element in the list declares which pipette it addresses.

N.B.: Python uses 0-based indexing! And therefore so does PyLabRobot.

N.B.2: Physical pipettes are counted from back to front, starting at 0!

In [9]:
star_channel_idxs = [6, 7] # easier to see for you guys in the audience ;)

await star.pipettes.pick_up_tips(
    tip_spots=star_teaching_tiprack["A1:B1"],
    use_channels=star_channel_idxs
)

---
## Probing

### ... the Deck

move pipettes to the position you want to probe, then call ztouch probing method:

#### on Prep
The Prep has a handy little calibration block at a known position for calibration purposes:


In [10]:
calibration_block = prep.get_resource("Prep_calibration_block")
calibration_block

Resource(name='Prep_calibration_block', location=Coordinate(129.550, 183.730, 000.000), size_x=6.0, size_y=6.0, size_z=22.0, category=calibration_block)

To `get_` its center_x, center_y, top position we can use PLR's resource model:

In [11]:
calibration_block_cct = calibration_block.get_location_wrt(
        other=prep.deck, # we always move motors relative to the deck
        x="c",
        y="c",
        z="t"
    )

calibration_block_cct

Coordinate(x=132.55, y=186.73, z=22.0)

In [12]:
await prep.pipettes.move_to_xy_positions(
    x = calibration_block_cct.x,
    ys = {prep_channel_idx : calibration_block_cct.y}
)

In [13]:
measured_z = await prep.pipettes.probe_z_using_ztouch(
    channel_idx=prep_channel_idx,
    # search_start_position: Optional[float] = None,
    # search_end_position: Optional[float] = None,
    # search_speed: float = 10.0,
    # push_force_pwm: Optional[int] = None,
    # end_tolerance: float = 1.2,
    # tip_len: Optional[float] = None,
    # allow_without_tip: bool = False,
    # post_detection_distance: float = 2.0,
    # move_to_safe_z_position_after: bool = False,
)
print(f"Prep: {measured_z=}")

await prep.pipettes.move_to_safe_z()

Prep: measured_z=22.0


#### (Almost) the same for STAR:

In [14]:
chosen_deck_coordinates = [
    Coordinate(400.0, 500.0, 245.0),
    Coordinate(400.0, 200.0, 245.0),
]
chosen_ys = [coord.y for coord in chosen_deck_coordinates]


await star.pipettes.move_to_xy_positions(
    x = chosen_deck_coordinates[0].x,
    ys = dict(zip(star_channel_idxs, chosen_ys))
)


measured_zs = await asyncio.gather(
    *(
        star.pipettes.probe_z_using_ztouch(
            channel=ch
        )
        for ch in star_channel_idxs)
  )
print(f"STARlet: {measured_zs=}")

await star.pipettes.move_to_safe_z()

STARlet: measured_zs=[None, None]


### ... in multiple locations 

In [15]:
await prep.pipettes.probe_deck_corner_z_offsets()

{'30.0,-2.5': 13.5,
 '30.0,375.0': 13.5,
 '235.0,-2.5': 13.5,
 '235.0,375.0': 13.5}

then we want to ensure we place the teaching needle back:

In [16]:
# via an explicit drop command:

# await prep.pipettes.drop_tips(
#     tip_spots=[prep_tipspot], # has to be a list
#     use_channels=[prep_channel_idx]
# )

# or the lazy way :P

await prep.pipettes.return_tips()

The PLR STAR(let) does not yet have this deck probing convience method.

...but we still have to return the teaching needles on the STARlets:

In [17]:
await star.pipettes.return_tips()

---
### ... tip presence

using the sleeve sensors of the pipettes we can check whether a tip exists at a tipspot.

i.e. we can verify the state of the resource model
...and correct it if the model is outdated!

But first, we need some tips and activate `set_tip_tracking`:

In [18]:
from pylabrobot.resources import set_tip_tracking
from pylabrobot.resources.hamilton import (
    hamilton_96_tiprack_50uL_NTR,
    hamilton_96_tiprack_300uL_NTR
)

set_tip_tracking(enabled=True)

then create new tipracks and assign them to the deck:

In [19]:
ntr_50ul_00 = prep.deck[1] = hamilton_96_tiprack_50uL_NTR(
    name="ntr_50ul_00",
    )

ntr_300ul_00 = prep.deck[7] = hamilton_96_tiprack_300uL_NTR(
    name="ntr_300ul_00",
    )

In [20]:
start_time = time.time()

tip_probing_results = await prep.pipettes.probe_tip_presence_via_pickup(
    tip_spots=ntr_50ul_00["G4:H4"]
)

end_time = time.time() - start_time

tip_probing_results, f"duraction: {end_time} sec"

({'ntr_50ul_00_tipspot_G4': True, 'ntr_50ul_00_tipspot_H4': True},
 'duraction: 0.0049479007720947266 sec')

#### Update if model is outdated

In [21]:
tip_probing_results = await prep.pipettes.probe_tip_presence_via_pickup(
    tip_spots=ntr_50ul_00["G1:H1"]
)

for spot_name, has_tip in tip_probing_results.items():
    print(spot_name, has_tip)

    if not has_tip:
        # the model said there was a tip,
        # the device found none:
        print(f" -> correct the model's tipstate")
        ntr_50ul_00.get_resource(spot_name).unassign_tip()

ntr_50ul_00_tipspot_G1 True
ntr_50ul_00_tipspot_H1 True


same on the STAR:

In [ ]:
from pylabrobot.lib.liquid_handling.tip_presence_probing import probe_tip_presence_via_pickup
from pylabrobot.resources.hamilton import hamilton_tip_carrier_L5_ntr_a00


tip_carrier = hamilton_tip_carrier_L5_ntr_a00(name="star_tip_carrier")
tip_carrier[0] = star_ntr_50ul_00 = hamilton_96_tiprack_50uL_NTR(name="star_ntr_50ul_00")
star.deck.assign_child_resource(tip_carrier, track=13)

In [ ]:
# the STAR has no `probe_tip_presence_via_pickup` of its own yet: the shared helper drives its calls
tip_probing_results = await probe_tip_presence_via_pickup(
    tip_spots=star_ntr_50ul_00["G1:H1"],
    use_channels=star_channel_idxs,
    pick_up_tips=lambda spots, channels: star.pipettes.pick_up_tips(spots, use_channels=channels),
    drop_tips=lambda spots, channels: star.pipettes.drop_tips(spots, use_channels=channels),
    sense_tip_presence=star.pipettes.sense_tip_presence,
)

for spot_name, has_tip in tip_probing_results.items():
    print(spot_name, has_tip)

    if not has_tip:
        print(f" -> correct the model's tipstate")
        star_ntr_50ul_00.get_resource(spot_name).unassign_tip()

star_ntr_50ul_00_tipspot_G1 True
star_ntr_50ul_00_tipspot_H1 True


## STAR only: carrier sensing

In [23]:
await star.autoload.sense_carrier_presence_on_deck()

[18]

In [25]:
await star.autoload.sense_carrier_presence_on_loading_tray()
# N.B.: this will move the autoload back and forth
# along the entire x-length of the device

[]

In [29]:
for track_idx in [40, 39, 41]:
    sense_result = await star.autoload.sense_carrier_presence_on_single_loading_tray_track(
        track=track_idx,
        park_after=False
    )
    print(f"{track_idx}: track present? -> {sense_result}")

await star.autoload.park()

40: track present? -> False
39: track present? -> False
41: track present? -> False


---
# Shutdown

In [17]:
await star.stop()

In [18]:
await prep.stop()

2026-09-27 11:31:26,133 - pylabrobot.hamilton.transport.tcp.tcp - INFO - Hamilton TCP client stopped


In [19]:
await viewer.stop()